In [0]:
from delta.tables import DeltaTable

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

from pyspark.sql.functions import current_timestamp


# ============================================================
# 1. CONFIGURATION
# ============================================================

control_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)

# Change this value depending on the pipeline
pipeline_name = "customers_silver"


# ============================================================
# 2. CONTROL TABLE SCHEMA
# ============================================================

control_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("last_processed_timestamp", TimestampType(), True),
    StructField("last_run_id", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])


# ============================================================
# 3. CREATE CONTROL TABLE IF IT DOES NOT EXIST
# ============================================================

if not DeltaTable.isDeltaTable(spark, control_path):

    empty_df = spark.createDataFrame(
        [],
        control_schema
    )

    (
        empty_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(control_path)
    )

    print("Control table created.")

else:

    print("Control table already exists.")


# ============================================================
# 4. READ CONTROL TABLE
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)


# ============================================================
# 5. INITIALIZE CURRENT PIPELINE IF MISSING
# ============================================================

if control_df.filter(
    control_df.pipeline_name == pipeline_name
).count() == 0:

    initial_pipeline_df = (
        spark.createDataFrame(
            [
                (
                    pipeline_name,
                    None,
                    None,
                    None
                )
            ],
            control_schema
        )
        .withColumn(
            "updated_at",
            current_timestamp()
        )
    )

    (
        initial_pipeline_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )

    print(
        f"{pipeline_name} initialized."
    )

else:

    print(
        f"{pipeline_name} already exists."
    )


# ============================================================
# 6. FINAL CHECK
# ============================================================

final_control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)

display(final_control_df)

Control table already exists.
customers_silver already exists.


pipeline_name,last_processed_timestamp,last_run_id,updated_at
customers_silver,2026-10-07T17:31:29.384Z,"CUSTOMERS_REBUILD_Column<'date_format(current_timestamp(), yyyyMMddHHmmss)'>",2026-10-07T17:32:01.236Z
gold_product_sales,2026-10-07T17:31:26.992Z,GOLD_PRODUCT_SALES_20261007173126,2026-10-07T17:32:44.445Z
gold_daily_sales,2026-10-07T17:31:26.992Z,GOLD_DAILY_SALES_20261007173126,2026-10-07T17:32:30.894Z
gold_customer_sales,2026-10-07T17:31:26.992Z,GOLD_CUSTOMER_SALES_RUN,2026-10-07T17:32:44.357Z
gold_market_sales,2026-10-07T17:31:26.992Z,GOLD_MARKET_SALES_RUN,2026-10-07T17:32:36.045Z
orders_silver,2026-10-07T21:53:20.000Z,RUN_b57ca01a,2026-10-07T17:32:01.690Z
products_silver,2026-10-07T17:31:26.972Z,null,2026-10-07T17:31:57.077Z
